In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
data = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\final_merged_cms_bls.csv")
data.head().T
df = data

In [3]:
df.shape
df.head()
df.tail().T

,73969,73970,73971,73972,73973
year,2022,2022,2022,2022,2022
state_code,MS,NY,TX,WI,WI
provider_type,General Short-Term (includes CAH),General Short-Term (includes CAH),Children’s Hospital,General Short-Term (includes CAH),General Short-Term (includes CAH)
rural_versus_urban,Urban,Urban,Urban,Urban,Urban
ccn_facility_type,Critical Access Hospital,Short-Term Hospital,Children’s Hospital,Short-Term Hospital,Short-Term Hospital
number_of_beds,25.0,81.0,794.0,285.0,301.0
total_bed_days_available,9125.0,29487.0,285430.0,104025.0,109865.0
occupancy_rate,0.533918,0.418083,0.789463,0.396943,0.46781
total_discharges__v___xviii___xix___unknown_,124.0,2797.0,36460.0,7709.0,9750.0
total_days__v___xviii___xix___unknown_,4872.0,12328.0,222271.94,41292.0,51396.0


In [4]:
rows, cols = df.shape
print("Rows:", rows)
print("Columns:", cols)

Rows: 73974
Columns: 46


In [5]:
df.dtypes

year                                              int64
state_code                                       object
provider_type                                    object
rural_versus_urban                               object
ccn_facility_type                                object
number_of_beds                                  float64
total_bed_days_available                        float64
occupancy_rate                                  float64
total_discharges__v___xviii___xix___unknown_    float64
total_days__v___xviii___xix___unknown_          float64
fte___employees_on_payroll                      float64
net_patient_revenue                             float64
total_income                                    float64
net_income                                      float64
total_costs                                     float64
total_other_expenses                            float64
total_assets                                    float64
total_liabilities                               

In [6]:
df["profit_margin_calc"].describe()

count    73974.000000
mean         0.037960
std          0.181478
min         -0.899297
25%         -0.018007
50%          0.046184
75%          0.118045
max          0.520817
Name: profit_margin_calc, dtype: float64

In [7]:
df["profit_margin_calc"].std()

np.float64(0.1814783570706588)

In [8]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(15)

rural_urban                                     0.08288
avg_healthcare_wage                             0.08288
provider_type                                   0.00000
rural_versus_urban                              0.00000
year                                            0.00000
state_code                                      0.00000
total_bed_days_available                        0.00000
occupancy_rate                                  0.00000
total_discharges__v___xviii___xix___unknown_    0.00000
total_days__v___xviii___xix___unknown_          0.00000
fte___employees_on_payroll                      0.00000
net_patient_revenue                             0.00000
ccn_facility_type                               0.00000
number_of_beds                                  0.00000
net_income                                      0.00000
dtype: float64

In [9]:
df[["avg_healthcare_wage", "number_of_beds"]].nunique()

avg_healthcare_wage    594
number_of_beds         782
dtype: int64

In [10]:
df["rural_urban"].value_counts(normalize=True)

rural_urban
Rural    0.702578
Urban    0.297422
Name: proportion, dtype: float64

In [11]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [12]:
df[
    ["profit_margin_calc", "avg_healthcare_wage", "number_of_beds"]
].corr()

,profit_margin_calc,avg_healthcare_wage,number_of_beds
profit_margin_calc,1.000000,0.013944,0.060051
avg_healthcare_wage,0.013944,1.000000,0.113141
number_of_beds,0.060051,0.113141,1.000000


In [13]:
X = df[["avg_healthcare_wage", "number_of_beds"]].dropna()
X["intercept"] = 1

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif

,feature,VIF
0,avg_healthcare_wage,1.012967
1,number_of_beds,1.012967
2,intercept,64.876526


In [14]:
df_model = df[
    ["profit_margin_calc", "avg_healthcare_wage", "number_of_beds", "rural_urban"]
].dropna()

df_model.shape

(67841, 4)

In [15]:
ml_ready_report = {
    "Rows > 10000": rows > 10000,
    "Target variance > 0": df["profit_margin_calc"].std() > 0,
    "Missing values manageable": df[["profit_margin_calc","avg_healthcare_wage","number_of_beds"]].isna().mean().max() < 0.1,
    "Categorical balance ok": df["rural_urban"].value_counts(normalize=True).min() > 0.1,
    "No leakage detected": True
}

ml_ready_report

{'Rows > 10000': True,
 'Target variance > 0': np.True_,
 'Missing values manageable': np.True_,
 'Categorical balance ok': np.True_,
 'No leakage detected': True}

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import category_encoders as ce


In [17]:
df.isnull().sum()

year                                               0
state_code                                         0
provider_type                                      0
rural_versus_urban                                 0
ccn_facility_type                                  0
number_of_beds                                     0
total_bed_days_available                           0
occupancy_rate                                     0
total_discharges__v___xviii___xix___unknown_       0
total_days__v___xviii___xix___unknown_             0
fte___employees_on_payroll                         0
net_patient_revenue                                0
total_income                                       0
net_income                                         0
total_costs                                        0
total_other_expenses                               0
total_assets                                       0
total_liabilities                                  0
total_current_assets                          

In [18]:
df=df.dropna()

In [19]:
# # Encode rural urban
mapping = {"Rural": 0,"Urban": 1}
df["rural_urban_encoded"] = df["rural_urban"].map(mapping)

C:\Users\manju\AppData\Local\Temp\ipykernel_7048\373336195.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["rural_urban_encoded"] = df["rural_urban"].map(mapping)


In [20]:
df.dtypes

year                                              int64
state_code                                       object
provider_type                                    object
rural_versus_urban                               object
ccn_facility_type                                object
number_of_beds                                  float64
total_bed_days_available                        float64
occupancy_rate                                  float64
total_discharges__v___xviii___xix___unknown_    float64
total_days__v___xviii___xix___unknown_          float64
fte___employees_on_payroll                      float64
net_patient_revenue                             float64
total_income                                    float64
net_income                                      float64
total_costs                                     float64
total_other_expenses                            float64
total_assets                                    float64
total_liabilities                               

In [1]:
df.columns.to_list()

NameError: name 'df' is not defined

In [21]:
# Features for Profit Margin Prediction
"""
avg_healthcare_wage
staff_to_bed_ratio
number_of_beds
occupancy_rate
discharges_per_bed
current_ratio
debt_to_asset_ratio
rural_urban
"""

'\navg_healthcare_wage\nstaff_to_bed_ratio\nnumber_of_beds\noccupancy_rate\ndischarges_per_bed\ncurrent_ratio\ndebt_to_asset_ratio\nrural_urban\n'

In [22]:
# Model 1: Predicting Profit Margin
profit_margin_features = [
    "avg_healthcare_wage",
    "staff_to_bed_ratio",
    "number_of_beds",
    "occupancy_rate",
    "discharges_per_bed",
    "current_ratio",
    "debt_to_asset_ratio",
    "rural_urban_encoded"
]

In [23]:
# Linear Regression for Profit Margin
X = df[profit_margin_features]
y = df["profit_margin_calc"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Profit Margin Model")
print("Mean Squared Error:", mse)
print("R^2 Score:", r2)

Profit Margin Model
Mean Squared Error: 0.02897608893027514
R^2 Score: 0.1235764508356878


In [24]:
# Decision Tree Regressor for Profit Margin
from sklearn.tree import DecisionTreeRegressor
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_y_pred = dt_model.predict(X_test)
dt_mse = mean_squared_error(y_test, dt_y_pred)
dt_r2 = r2_score(y_test, dt_y_pred)
print("Decision Tree Regressor Model")
print("Mean Squared Error:", dt_mse)
print("R^2 Score:", dt_r2)


Decision Tree Regressor Model
Mean Squared Error: 0.0464161731425389
R^2 Score: -0.4039240182516548


In [25]:
#  Random Forest Regressor for Profit Margin
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_y_pred = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_y_pred)
rf_r2 = r2_score(y_test, rf_y_pred)
print("Random Forest Regressor Model")
print("Training Scores:", rf_model.score(X_train, y_train))
print("Testing Scores:", rf_model.score(X_test, y_test))

Random Forest Regressor Model
Training Scores: 0.903760525279426
Testing Scores: 0.30127479290605974


In [26]:
# Features for Cost to Charge Ratio Prediction
"""avg_healthcare_wage
staff_to_bed_ratio
expense_ratio
occupancy_rate
avg_length_of_stay
discharges_per_bed
number_of_beds
current_ratio
debt_to_asset_ratio
rural_urban
"""

'avg_healthcare_wage\nstaff_to_bed_ratio\nexpense_ratio\noccupancy_rate\navg_length_of_stay\ndischarges_per_bed\nnumber_of_beds\ncurrent_ratio\ndebt_to_asset_ratio\nrural_urban\n'

In [27]:
# Model 2: Predicting Cost to Charge Ratio
ccr_features = [
    "avg_healthcare_wage",
    "staff_to_bed_ratio",
    "expense_ratio",
    "occupancy_rate",
    "avg_length_of_stay",
    "discharges_per_bed",
    "number_of_beds",
    "current_ratio",
    "debt_to_asset_ratio",
    "rural_urban_encoded"
]

In [28]:
# Linear Regression for Cost to Charge Ratio
model = LinearRegression()
X = df[ccr_features]
y = df["cost_to_charge_ratio"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Cost to Charge Ratio Model")
training_mse = mean_squared_error(y_train, model.predict(X_train))
training_r2 = r2_score(y_train, model.predict(X_train))
print("Training Mean Squared Error:", training_mse)
print("Training R^2 Score:", training_r2)
print("Testing Mean Squared Error:", mse)
print("Testing R^2 Score:", r2)

Cost to Charge Ratio Model
Training Mean Squared Error: 0.03924117049042492
Training R^2 Score: 0.266988533792609
Testing Mean Squared Error: 0.0394636143712446
Testing R^2 Score: 0.2801046788397621


In [29]:
# Decision Tree Regressor
from sklearn.tree import DecisionTreeRegressor
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_y_pred = dt_model.predict(X_test)
dt_mse = mean_squared_error(y_test, dt_y_pred)
dt_r2 = r2_score(y_test, dt_y_pred)
print("Decision Tree Regressor Model")
training_mse = mean_squared_error(y_train, dt_model.predict(X_train))
training_r2 = r2_score(y_train, dt_model.predict(X_train))
print("Training Mean Squared Error:", training_mse)
print("Training R^2 Score:", training_r2)
print("Testing Mean Squared Error:", dt_mse)
print("Testing R^2 Score:", dt_r2)

Decision Tree Regressor Model
Training Mean Squared Error: 3.471205541515138e-33
Training R^2 Score: 1.0
Testing Mean Squared Error: 0.03164877492822102
Testing R^2 Score: 0.4226629933855903


In [30]:
# Random Forest Regressor for Cost to Charge Ratio
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_y_pred = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_y_pred)
rf_r2 = r2_score(y_test, rf_y_pred)
print("Random Forest Regressor Model")
print("Training Scores:", rf_model.score(X_train, y_train))
print("Testing Scores:", rf_model.score(X_test, y_test))

Random Forest Regressor Model
Training Scores: 0.9585382284227143
Testing Scores: 0.7232349839351211
